# <span class="nb-only" data-num="0."></span>Lab 02: Data Mining and Visualization

---

**Group 2** 

23127026 - Nguyễn Đỗ Bảo \
23127118 - Lê Nguyên Thảo \
23127144 - Đinh Đại Vũ \
23127427 - Vũ Hoàng Minh \
23127447 - Nguyễn Thanh Owen

---

# <span class="nb-only" data-num="1."></span>Basic Data Analysis

## <span class="nb-only" data-num="1.1"></span>Dataset Overview
The dataset utilized in this project is extracted from the **World Development Indicators (WDI)** by the World Bank. It provides a comprehensive collection of indicators regarding economic, social, and environmental development. This specific dataset focuses on **Viet Nam and Thailand** spanning a 24-year period from **2000 to 2023**. It serves as the foundation for exploring trends and relationships across five key domains: Macroeconomics, Healthcare, Demographics, Labor, and Globalization.

## <span class="nb-only" data-num="1.2"></span>Data Structure Description
Before any manipulation, we will load the dataset to examine its initial structure, including the number of records and available fields. The data is in a **panel (country–year) format**, where each row represents a specific country in a specific year, and each column represents a distinct development indicator.

<style>
.nb-only {
  display: none;
}
@media screen {
  .nb-only {
    display: inline;
  }
  .nb-only::before {
    content: attr(data-num) " ";
  }
}
@media print {
  .nb-only {
    display: none !important;
  }
}
</style>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from matplotlib.patches import Patch

In [ ]:
# Load the dataset
df = pd.read_csv('data/data.csv')

# Display basic structure information
print(f"Total records (rows): {df.shape[0]}")
print(f"Total fields (columns): {df.shape[1]}")

print("\n--- Column Information ---")
df.info()

In [ ]:
print("\n--- First 5 Records ---")
df.head()

## <span class="nb-only" data-num="1.3"></span>Initial Exploratory Data Analysis (EDA)
Before any cleaning or transformation, we perform EDA on the raw dataset to understand its current state and identify what needs to be fixed. The process consists of the following steps:

1. Descriptive Statistics: Compute summary statistics (`count`, `mean`, `std`, `min`, `max`, quartiles) on all numeric indicator columns to gauge value ranges and spot anomalies.
2. Missing Value Check: Quantify missing entries — both true `NaN` values and the World Bank placeholder string `..` — across every column, then visualize the missingness pattern.
3. Categorical Breakdown: Examine how many unique countries exist, list them, and verify the expected temporal coverage (countries × years = total rows).
4. Temporal Coverage Check: For each year, count how many valid (non-missing) observations exist across all indicator columns to reveal whether recent years suffer from reporting lags.
5. Value Distribution: Inspect the distribution of raw numeric values across indicator columns to detect skewness, outliers, or data-entry issues.

The subsection titles below use notebook-only numbering (hidden in PDF export).

### <span class="nb-only" data-num="1.3.1"></span>Descriptive Statistics

In [ ]:
# Identify indicator columns (everything except metadata columns)
meta_cols = ['Time', 'Time Code', 'Country Name', 'Country Code']
indicator_cols = [c for c in df.columns if c not in meta_cols]

# Convert '..' placeholders to NaN temporarily so pandas can compute numeric stats
df_numeric = df[indicator_cols].replace('..', np.nan).apply(pd.to_numeric, errors='coerce')

print("Descriptive Statistics (across all indicator columns)")
df_numeric.describe().T

### <span class="nb-only" data-num="1.3.2"></span>Missing Value Check

In [ ]:
# Count both true NaN and the '..' placeholder per column
placeholder_counts = (df == '..').sum()
nan_counts = df.isna().sum()
total_missing = placeholder_counts + nan_counts

missing_df = pd.DataFrame({
    'NaN count': nan_counts,
    "'..'' count": placeholder_counts,
    'Total missing': total_missing,
    'Missing %': (total_missing / len(df) * 100).round(2)
})

print("Missing Value Summary")
print(missing_df.to_string())

# Visualize missingness for indicator columns only
fig, ax = plt.subplots(figsize=(14, 5))
indicator_labels = [c.split('[')[0].strip()[:40] for c in indicator_cols]  # Short labels
bars_placeholder = [(df[c] == '..').sum() for c in indicator_cols]
bars_nan = [df[c].isna().sum() for c in indicator_cols]

x = range(len(indicator_cols))
ax.bar(x, bars_placeholder, label="'..' placeholder", color='#e74c3c')
ax.bar(x, bars_nan, bottom=bars_placeholder, label='True NaN', color='#95a5a6')
ax.set_xticks(x)
ax.set_xticklabels(indicator_labels, rotation=90, fontsize=7)
ax.set_ylabel('Number of Missing Entries')
ax.set_title('Missing Values per Indicator Column (out of {} rows)'.format(len(df)))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Identify rows with missing data (NaN or '..' placeholder)
df_check = df[indicator_cols].replace('..', np.nan)
missing_per_row = df_check.isna().sum(axis=1)

# Build a summary table: Country, Year, number of missing indicators
row_missing_df = pd.DataFrame({
    'Country': df['Country Name'],
    'Year': df['Time'],
    'Missing Indicators': missing_per_row,
    'Total Indicators': len(indicator_cols),
    'Missing %': (missing_per_row / len(indicator_cols) * 100).round(1)
})

# Show only rows that have at least one missing value
rows_with_missing = row_missing_df[row_missing_df['Missing Indicators'] > 0].sort_values(
    by='Missing Indicators', ascending=False
)

print(f"Rows with missing data: {len(rows_with_missing)} / {len(df)}")
print(f"Rows with complete data: {len(df) - len(rows_with_missing)} / {len(df)}\n")
display(rows_with_missing)

# Show which specific indicators are missing for the worst rows
print("\n--- Detailed Missing Indicators for Rows with Most Gaps ---")
top_missing = rows_with_missing.head(10)
for idx in top_missing.index:
    country = df.loc[idx, 'Country Name']
    year = df.loc[idx, 'Time']
    missing_cols = df_check.columns[df_check.loc[idx].isna()].tolist()
    short_names = [c.split('[')[0].strip()[:50] for c in missing_cols]
    print(f"\n{country} ({year}) — {len(missing_cols)} missing:")
    for name in short_names:
        print(f"  • {name}")

After performing missing value check, we can confirm that there are no missing values. Therefore, missing value handling is not necessary anymore.

### <span class="nb-only" data-num="1.3.3"></span>Categorical Breakdown

In [ ]:
n_countries = df['Country Name'].nunique()
n_years = df['Time'].nunique()
n_indicators = len(indicator_cols)

print("--- Categorical Breakdown ---")
print(f"Unique countries  : {n_countries}")
print(f"Unique years      : {n_years} ({df['Time'].min()} – {df['Time'].max()})")
print(f"Indicator columns : {n_indicators}")
print(f"Expected rows (countries × years): {n_countries} × {n_years} = {n_countries * n_years}")
print(f"Actual rows: {len(df)}")

print("\n--- Countries ---")
print(df['Country Name'].value_counts().to_string())

print("\n--- Indicators ---")
for i, col in enumerate(indicator_cols, 1):
    short_name = col.split('[')[0].strip()
    print(f"  {i:2d}. {short_name}")

### <span class="nb-only" data-num="1.3.4"></span>Temporal Coverage Check

In [ ]:
# Count valid (non-missing) observations per year across all indicator columns
years = sorted(df['Time'].unique())
valid_counts = []
for year in years:
    year_data = df[df['Time'] == year][indicator_cols].replace('..', np.nan)
    year_data = year_data.apply(pd.to_numeric, errors='coerce')
    valid = year_data.notna().sum().sum()  # Total valid cells for that year
    valid_counts.append(valid)

max_possible = n_countries * n_indicators  # Max valid cells per year

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#2ecc71' if v > max_possible * 0.7 else '#e67e22' if v > max_possible * 0.4 else '#e74c3c' for v in valid_counts]
bars = ax.bar([str(y) for y in years], valid_counts, color=colors, edgecolor='white')
ax.axhline(y=max_possible, color='black', linestyle='--', alpha=0.5, label=f'Max possible ({max_possible})')
ax.set_xlabel('Year')
ax.set_ylabel('Valid Observations')
ax.set_title('Temporal Coverage: Valid (non-missing) Entries per Year')
ax.legend()

# Annotate bars with percentages
for bar, v in zip(bars, valid_counts):
    pct = v / max_possible * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{pct:.0f}%', ha='center', va='bottom', fontsize=7)

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

The temporal coverage check also indicates the same information as the missing value check.

### <span class="nb-only" data-num="1.3.5"></span>Value Distribution

In [ ]:
# Stack all indicator values into a single series to see the overall distribution
all_values = df_numeric.stack().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 5a. Histogram
axes[0].hist(all_values, bins=50, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of All Numeric Values')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].set_yscale('log')  # Log scale due to extreme range differences across indicators

# 5b. Box plot per indicator (sampled to show spread)
df_numeric.boxplot(ax=axes[1], rot=90, flierprops=dict(marker='.', markersize=2, alpha=0.4))
axes[1].set_title('Box Plot per Indicator Column')
axes[1].set_ylabel('Value')
axes[1].set_yscale('log')
axes[1].set_xticklabels([c.split('[')[0].strip()[:20] for c in indicator_cols], fontsize=6)

plt.tight_layout()
plt.show()

print(f"\nOverall value range: [{all_values.min():.4f}, {all_values.max():.4f}]")

The range confirms that the dataset have features with varying value scales. Therefore, some method of normalization or standardization can be applied.

## <span class="nb-only" data-num="1.4"></span>Data Preprocessing
The initial exploration reveals that the dataset requires cleaning to be suitable for visualization and statistical analysis. Since the data is already in a panel format (one row per country-year), the preprocessing is straightforward:

1. Rename columns: Strip the World Bank indicator codes (e.g., `[NY.GDP.PCAP.KN]`) from column names for readability.
2. Standardize the 'Year' column: Rename `Time` to `Year` and ensure it is an integer type.
3. Data type conversion: Ensure all indicator columns are cast to a numerical format (float).
4. Drop redundant columns: Remove `Time Code` as it duplicates the `Year` information.
5. Drop redundant columns: Remove `Country Code` as it duplicates the `Country Name` information.

In [ ]:
# 1. Replace '..' with NaN
df_cleaned = df.replace('..', np.nan)

# 2. Rename columns: strip World Bank indicator codes [XX.XXX.XXX]
df_cleaned.columns = [re.sub(r'\s*\[.*?\]\s*$', '', col).strip() for col in df_cleaned.columns]

# 3. Standardize Year column
df_cleaned = df_cleaned.rename(columns={'Time': 'Year'})
df_cleaned['Year'] = df_cleaned['Year'].astype(int)

# 4. Drop redundant columns
df_cleaned = df_cleaned.drop(columns=['Time Code', 'Country Code'], errors='ignore')

# 5. Convert all indicator columns to float
meta_cols_clean = ['Year', 'Country Name']
indicator_cols_clean = [c for c in df_cleaned.columns if c not in meta_cols_clean]
df_cleaned[indicator_cols_clean] = df_cleaned[indicator_cols_clean].apply(pd.to_numeric, errors='coerce')

# Assign to df_final for consistency with downstream cells
df_final = df_cleaned.copy()

print(f"Cleaned dataset shape: {df_final.shape[0]} rows, {df_final.shape[1]} columns")
display(df_final.head())

# <span class="nb-only" data-num="2."></span>Define Objectives and Select Data Fields

In this section, we define 15 specific analysis objectives grouped into five key domains. For each objective, we explicitly state the selected WDI metrics, explain their meanings, justify their selection, and examine the underlying relationships between these fields. The analysis focuses on **Viet Nam and Thailand** over the period **2000–2023**.


## <span class="nb-only" data-num="2.1"></span>Macroeconomics

**Target 1: Assess the economic growth.**

* **Metrics & Meaning:**
  * `GDP growth (annual %)`: The annual percentage change in a country's total economic output (Gross Domestic Product), adjusted for inflation. A positive value indicates economic expansion, while a negative value signals contraction.
* **Rationale & Relationships:** GDP growth is the most direct measure of macroeconomic momentum. Tracking it over time reveals the pace and consistency of economic expansion, highlights the impact of global crises (e.g., the 1997 Asian financial crisis aftermath, the 2008 global recession, the 2020 COVID-19 pandemic), and enables a direct comparison of growth trajectories between Viet Nam and Thailand.

**Target 2: Analyze the structural shift from agriculture to industry/services.**

* **Metrics & Meaning:**
  * `Agriculture, forestry, and fishing, value added (% of GDP)`: The net output of the agricultural sector as a proportion of the total national economy.
  * `Industry (including construction), value added (% of GDP)`: The net output of the industrial sector (including manufacturing, mining, and construction) as a proportion of the economy.
  * `Services, value added (% of GDP)`: The net output of the services sector (including retail, finance, and government) as a proportion of the economy.
* **Rationale & Relationships:** Economic modernization typically involves a transition away from an agriculture-based economy toward manufacturing and services. By tracking the proportional changes of these three core sectors simultaneously, you can observe this structural transformation over time and compare the developmental maturity of Viet Nam versus Thailand.

**Target 3: Evaluate the standard of living.**

* **Metrics & Meaning:**
  * `GDP per capita, PPP (constant 2021 international $)`: The average economic output per person, converted to international dollars using purchasing power parity (PPP). This standardizes for cross-country price differences, making it the most reliable single metric for comparing real living standards at constant 2021 price levels.
* **Rationale & Relationships:** Aggregate GDP can be misleading for countries with different population sizes. PPP-adjusted GDP per capita normalizes for both population and local price levels, enabling a fair comparison of whether economic growth is translating into improved material welfare for the average citizen in each country.


## <span class="nb-only" data-num="2.2"></span>Globalization

**Target 4: Assess the openness of the economy.**

* **Metrics & Meaning:**
  * `Trade (% of GDP)`: The sum of all exports and imports of goods and services measured as a share of gross domestic product. It is the broadest single indicator of how integrated an economy is with the rest of the world.
  * `GDP growth (annual %)`: The annual rate of economic expansion.
* **Rationale & Relationships:** A very high trade-to-GDP ratio indicates deep integration into global supply chains. Tracking this alongside annual GDP growth allows you to observe whether a country's economic expansion is disproportionately vulnerable to external global shocks (like international recessions or supply chain collapses) rather than being driven by resilient domestic demand. Comparing Viet Nam and Thailand reveals which economy is more exposed to global volatility.

**Target 5: Measure global trade integration.**

* **Metrics & Meaning:**
  * `Exports of goods and services (% of GDP)`: The total value of goods and services sold to the rest of the world, as a share of the domestic economy.
  * `Imports of goods and services (% of GDP)`: The total value of goods and services purchased from the rest of the world, as a share of the domestic economy.
* **Rationale & Relationships:** While Target 4 uses aggregate trade openness, this target decomposes it into export and import components. Tracking both separately reveals whether each country is a net exporter or net importer, how the trade balance shifts over time, and whether growth is being driven by foreign demand (exports) or domestic consumption (imports).

**Target 6: Evaluate Foreign Direct Investment (FDI) as a growth catalyst.**

* **Metrics & Meaning:**
  * `Foreign direct investment, net inflows (% of GDP)`: The net amount of cross-border investment made by foreign entities directly into domestic businesses, as a share of the economy. It reflects private-sector confidence in a country's market potential.
  * `GDP growth (annual %)`: The annual rate of economic expansion.
* **Rationale & Relationships:** By analyzing these two variables together, you can determine if external capital injections (FDI) have a statistically observable correlation with domestic economic expansion. This is particularly relevant for Viet Nam and Thailand, both of which have actively pursued FDI-driven industrialization strategies to fuel growth.

**Target 7: Evaluate the transition from foreign aid to private investment.**

* **Metrics & Meaning:**
  * `Net ODA received (% of GNI)`: The total net official development assistance (foreign aid from governments and multilateral organizations) received by a country, expressed as a share of its Gross National Income. It reflects the degree to which a nation depends on external aid for its development financing.
  * `Foreign direct investment, net inflows (% of GDP)`: The net amount of cross-border investment made by foreign entities directly into domestic businesses, as a share of the economy. It represents market-driven private capital attracted by economic opportunity.
* **Rationale & Relationships:** ODA and FDI represent two fundamentally different sources of external financing: public aid versus private investment. As economies mature, they are expected to graduate from aid dependency toward attracting market-driven capital. Comparing these two metrics over time for Viet Nam and Thailand reveals how each country has navigated this transition — with declining ODA reliance and rising FDI inflows signaling growing economic self-sufficiency and investor confidence.


## <span class="nb-only" data-num="2.3"></span>Environment

**Target 8: Evaluate the impact of urban expansion on national forest reserves.**

* **Metrics & Meaning:**
  * `Urban population growth (annual %)`: The annual rate of change in the number of people living in urban areas.
  * `Forest area (% of land area)`: The proportion of a country's total land area that is covered by natural or planted forests.
* **Rationale & Relationships:** Rapid urbanization often requires clearing land for housing, infrastructure, and expanding city perimeters. By plotting the urban growth rate against forest cover percentage, you can analytically determine if there is a direct, negative correlation — identifying urban expansion as a potential catalyst for deforestation in either country.

**Target 9: Assess drivers of deforestation.**

* **Metrics & Meaning:**
  * `Forest area (% of land area)`: The proportion of land covered by forests.
  * `Agricultural land (% of land area)`: The proportion of land designated for arable crops, permanent crops, and permanent pastures.
* **Rationale & Relationships:** Agricultural expansion is a well-documented driver of land clearing. By plotting these two metrics against each other over a time-series, you can determine if a rising share of agricultural land strongly correlates with a declining share of forest area, analytically testing whether agriculture is the primary driver of deforestation in each country.


## <span class="nb-only" data-num="2.4"></span>Healthcare & Demographics

**Target 10: Evaluate national prioritization of healthcare.**

* **Metrics & Meaning:**
  * `Current health expenditure (% of GDP)`: The total financial resources — from both public and private sources — spent on healthcare goods and services as a share of the national economy. It measures a country's overall financial commitment to its healthcare system.
* **Rationale & Relationships:** Healthcare expenditure as a share of GDP is a direct measure of how much economic priority a nation places on population health. Tracking this metric over time reveals whether each government is scaling up investment in healthcare infrastructure or if health remains underfunded relative to economic capacity. Comparing Viet Nam and Thailand provides insight into how countries at different income levels allocate resources to health.

**Target 11: Examine gender composition in national demographics.**

* **Metrics & Meaning:**
  * `Sex ratio at birth (male births per female births)`: The number of male births for every single female birth. The biological norm is approximately 1.05, but this ratio can be skewed by societal practices such as son preference or sex-selective interventions.
  * `Population, female (% of total population)`: The share of the total population that is female. It captures the cumulative effect of birth ratios, differential mortality, and migration patterns on the overall gender balance.
* **Rationale & Relationships:** The sex ratio at birth reflects whether biological norms hold or are being distorted by cultural factors. When paired with the overall female population share, these metrics reveal whether gender imbalances at birth persist, self-correct, or amplify over the life course — influenced by differential life expectancy, migration, and social factors. Tracking these together helps assess demographic balance in both countries.

**Target 12: Analyze the national demographic transition.**

* **Metrics & Meaning:**
  * `Birth rate, crude (per 1,000 people)`: The number of live births occurring during the year, per 1,000 population.
  * `Death rate, crude (per 1,000 people)`: The number of deaths occurring during the year, per 1,000 population.
* **Rationale & Relationships:** These metrics are the foundation of the Demographic Transition Model. By plotting crude birth rates alongside crude death rates over time, you can classify each country's developmental stage. The gap between these two lines reveals whether a nation is experiencing rapid population expansion (high birth, falling death), stabilization (both converging), or impending population decline (births falling below deaths).



## <span class="nb-only" data-num="2.5"></span>Labor & Workforce

**Target 13: Identify gender disparity in economic participation.**

* **Metrics & Meaning:**
  * `Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)`: The proportion of the working-age female population actively employed or seeking employment.
  * `Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)`: The proportion of the working-age male population actively employed or seeking employment.
* **Rationale & Relationships:** By placing the female participation rate directly alongside the male baseline, you establish a strictly quantifiable measure of structural gender inequality. Tracking the numeric gap between these two metrics over time provides an objective view of each country's structural barriers and its progress toward full economic integration of its working-age population.

**Target 14: Analyze the structural shift in employment across economic sectors.**

* **Metrics & Meaning:**
  * `Employment in agriculture (% of total employment) (modeled ILO estimate)`: The share of workers employed in farming, forestry, and fishing.
  * `Employment in industry (% of total employment) (modeled ILO estimate)`: The share of workers employed in manufacturing, mining, construction, and utilities.
  * `Employment in services (% of total employment) (modeled ILO estimate)`: The share of workers employed in trade, transport, finance, public administration, and other service activities.
* **Rationale & Relationships:** Employment composition reveals how labor migrates between sectors during economic development. While GDP composition (Target 2) shows the *value* each sector contributes, employment shows where *people* actually work. Comparing the two exposes productivity gaps — sectors that contribute disproportionately high GDP relative to their employment share have higher labor productivity. A stacked area chart tracks this structural transformation over time for both countries.

**Target 15: Assess the macroeconomic drivers of unemployment.**

* **Metrics & Meaning:**
  * `Unemployment, total (% of total labor force) (modeled ILO estimate)`: The share of the total labor force that is without work but available for and seeking employment. It is the primary indicator of labor market health.
  * `Inflation, consumer prices (annual %)`: The annual percentage change in the cost to the average consumer of acquiring a basket of everyday goods and services. It reflects purchasing power erosion and macroeconomic price stability.
  * `GDP growth (annual %)`: The annual rate of economic expansion, measuring how fast the economy's total output is growing or contracting.
* **Rationale & Relationships:** These three metrics form the core of macroeconomic stability analysis. Economic theory (Okun's Law) posits that GDP growth should reduce unemployment, while the Phillips Curve suggests a trade-off between unemployment and inflation. By tracking all three simultaneously, you can determine whether economic expansion is creating jobs (job-rich growth), whether price stability is being maintained, and how these dynamics differ between Viet Nam and Thailand.

# <span class="nb-only" data-num="3."></span>Data Visualization

In this section, we execute visualizations for each of the 15 defined targets using `matplotlib` and `seaborn`. Each visualization is designed following core principles of visual grammar:

- **Perceptual hierarchy** (Cleveland & McGill): Position on a common scale is the most accurately decoded visual channel, followed by length, angle, and area. We prioritize position-based encodings.
- **Data-ink ratio** (Tufte): Maximize the share of ink devoted to data; minimize non-data elements.
- **Small multiples** (Tufte): When comparing two countries across multiple metrics, side-by-side panels with shared axes outperform overloaded single charts.
- **Bertin's visual variables**: We match data types to appropriate visual channels — position for quantitative values, color hue for nominal categories (country), line style for metric type.

**Color conventions used throughout:**
- **Viet Nam**: `#D9534F` (red) — consistent across all charts
- **Thailand**: `#5BC0DE` (blue) — consistent across all charts
- Crisis periods (2008–2009 Global Financial Crisis, 2020 COVID-19) are shaded in grey where relevant.

In [ ]:
# ── Global Setup ─────────────────────────────────────────────────────────────────
C_VN = '#D9534F'
C_TH = '#5BC0DE'
C_CRISIS = '#e0e0e0'

vn = df_final[df_final['Country Name'] == 'Viet Nam'].sort_values('Year').reset_index(drop=True)
th = df_final[df_final['Country Name'] == 'Thailand'].sort_values('Year').reset_index(drop=True)
years = vn['Year'].values

def shade_crises(ax):
    """Add shaded bands for major economic crises."""
    ax.axvspan(2007.5, 2009.5, color=C_CRISIS, alpha=0.4, zorder=0, label='_nolegend_')
    ax.axvspan(2019.5, 2021.5, color=C_CRISIS, alpha=0.4, zorder=0, label='_nolegend_')

print(f"Viet Nam: {len(vn)} rows | Thailand: {len(th)} rows | Years: {years[0]}–{years[-1]}")

## <span class="nb-only" data-num="3.1"></span>Macroeconomics

### <span class="nb-only" data-num="3.1.1"></span>Target 1: Assess the economic growth

**Chart type:** Line chart (two series on a common y-axis)

* **What:** GDP growth (annual %) for Viet Nam and Thailand, 2000–2023.
* **Why:** A line chart is the optimal choice for a single continuous variable over ordered time (Cleveland & McGill: position on a common scale is perceptually the most accurate). The slope of each line segment directly encodes the rate of change. Since both series share the same unit and scale, a single y-axis enables fair, distortion-free comparison.
* **How:** Country distinguished by color hue (nominal variable → color). Crisis periods shaded as contextual annotation. A zero-baseline reference line separates growth from contraction.

In [ ]:
# ── Target 1: GDP Growth ─────────────────────────────────────────────────────────
col = 'GDP growth (annual %)'

fig, ax = plt.subplots(figsize=(14, 5))
shade_crises(ax)

ax.plot(years, vn[col], color=C_VN, marker='o', markersize=4, linewidth=2, label='Viet Nam')
ax.plot(years, th[col], color=C_TH, marker='s', markersize=4, linewidth=2, label='Thailand')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)

ax.set_xlabel('Year')
ax.set_ylabel('GDP Growth (%)')
ax.set_title('Annual GDP Growth: Viet Nam vs Thailand (2000-2023)')
ax.legend()
ax.set_xlim(years[0] - 0.5, years[-1] + 0.5)
ax.grid(axis='y', alpha=0.3)

# Annotate extremes
for series, name, color in [(vn, 'VN', C_VN), (th, 'TH', C_TH)]:
    idx_min = series[col].idxmin()
    ax.annotate(f'{series.loc[idx_min, col]:.1f}%',
                xy=(series.loc[idx_min, 'Year'], series.loc[idx_min, col]),
                textcoords='offset points', xytext=(15, -5), fontsize=8,
                color=color, ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Consistent Growth vs Volatility:** Viet Nam exhibits relatively stable GDP growth, maintaining positive rates throughout 2000–2023. Thailand shows higher volatility with sharper contractions during major shock periods.
* **Crisis-period contrast:** During 2008–2009 and 2020, Thailand's downturns are visibly deeper in this dataset, while Viet Nam's growth decelerates but remains positive. This pattern is consistent with different exposure structures across the two economies.
* **Convergence in Recent Years:** Post-2015, both countries show growth rates clustering more often in the 3–7% range, suggesting increasing macroeconomic stabilization in both cases.

### <span class="nb-only" data-num="3.1.2"></span>Target 2: Analyze the structural shift from agriculture to industry/services

**Chart type:** Stacked area chart (small multiples — one panel per country)

* **What:** Agriculture, Industry, and Services value added (% of GDP) for both countries, 2000–2023.
* **Why:** This is a composition-over-time question: the three sectors sum to approximately 100% of GDP. A stacked area chart directly encodes the part-to-whole relationship, showing both individual sector trends and total composition simultaneously. Small multiples (one panel per country) prevent visual clutter and enable clean comparison (Tufte).
* **How:** Each sector encoded by color hue (categorical). Shared y-axis (0–100%) ensures fair cross-panel comparison. Sectors stacked in consistent order across both panels.

In [ ]:
# ── Target 2: Structural Shift (Original) ────────────────────────────────────────
c_agri = 'Agriculture, forestry, and fishing, value added (% of GDP)'
c_ind  = 'Industry (including construction), value added (% of GDP)'
c_srv  = 'Services, value added (% of GDP)'
sector_colors = {'Agriculture': '#66BB6A', 'Industry': '#42A5F5', 'Services': '#FFA726'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, (data, title) in zip(axes, [(vn, 'Viet Nam'), (th, 'Thailand')]):
    ax.stackplot(data['Year'],
                 data[c_agri], data[c_ind], data[c_srv],
                 labels=['Agriculture', 'Industry', 'Services'],
                 colors=[sector_colors['Agriculture'], sector_colors['Industry'], sector_colors['Services']],
                 alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylim(0, 105)
    ax.set_xlim(years[0], years[-1])
    ax.grid(axis='y', alpha=0.3)

    # Annotate end percentages for each sector
    for col_name, y_offset in [(c_agri, 0), (c_ind, data[c_agri].iloc[-1]),
                                (c_srv, data[c_agri].iloc[-1] + data[c_ind].iloc[-1])]:
        end_val = data[col_name].iloc[-1]
        ax.annotate(f'{end_val:.1f}%',
                    xy=(years[-1], y_offset + end_val / 2),
                    fontsize=8, ha='left', va='center', fontweight='bold')

axes[0].set_ylabel('Share of GDP (%)')

# Shared legend at the bottom
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.suptitle('Figure 2a — Economic Structure: Sectoral Composition of GDP (2000–2023)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Methodological note — Viet Nam's GDP composition gap (2010–2023):**

In the chart above, Viet Nam's three sectors visibly fail to sum to 100% from 2010 onward, leaving an ~8–11% gap at the top of the stacked area. This is **not** missing data — it reflects a change in Viet Nam's national accounting methodology. Starting around 2010, Viet Nam's General Statistics Office (GSO) revised its GDP calculation to report the three sectoral value-added shares **excluding "Product taxes less subsidies on products"**, which is counted separately as a fourth component. Thailand's national accounts, by contrast, fold this component into the three sectors, so its shares continue to sum to 100%.

To address this discrepancy, we present two additional views:
* **Figure 2b** adds the implicit "Product taxes less subsidies" component (computed as 100% minus the sum of the three reported sectors) to Viet Nam's stacked area, restoring the full 100% composition.
* **Figure 2c** normalizes Viet Nam's three reported sectors to sum to 100%, making the proportional comparison with Thailand directly comparable.

In [ ]:
# ── Target 2 — Figure 2b: With "Product Taxes less Subsidies" component ──────────
# Compute the implicit 4th component for Viet Nam
vn_tax = 100 - (vn[c_agri] + vn[c_ind] + vn[c_srv])
vn_tax = vn_tax.clip(lower=0)  # avoid tiny negatives from rounding

# Thailand has no gap, so its tax component is effectively zero
th_tax = 100 - (th[c_agri] + th[c_ind] + th[c_srv])
th_tax = th_tax.clip(lower=0)

sector_colors_4 = {
    'Agriculture': '#66BB6A', 'Industry': '#42A5F5',
    'Services': '#FFA726', 'Product taxes less subsidies': '#AB47BC'
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, (data, tax, title) in zip(axes, [(vn, vn_tax, 'Viet Nam'), (th, th_tax, 'Thailand')]):
    ax.stackplot(data['Year'],
                 data[c_agri], data[c_ind], data[c_srv], tax,
                 labels=['Agriculture', 'Industry', 'Services', 'Product taxes less subsidies'],
                 colors=[sector_colors_4['Agriculture'], sector_colors_4['Industry'],
                         sector_colors_4['Services'], sector_colors_4['Product taxes less subsidies']],
                 alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylim(0, 105)
    ax.set_xlim(years[0], years[-1])
    ax.grid(axis='y', alpha=0.3)

    # Annotate end percentages
    cumul = 0
    for val, lbl in [(data[c_agri].iloc[-1], 'Agri'),
                     (data[c_ind].iloc[-1], 'Ind'),
                     (data[c_srv].iloc[-1], 'Srv'),
                     (tax.iloc[-1], 'Tax')]:
        if val > 1:  # only annotate visible slices
            ax.annotate(f'{val:.1f}%',
                        xy=(years[-1], cumul + val / 2),
                        fontsize=8, ha='left', va='center', fontweight='bold')
        cumul += val

axes[0].set_ylabel('Share of GDP (%)')

# Shared legend at the bottom
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=10,
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.suptitle('Figure 2b — Sectoral Composition Including Product Taxes less Subsidies (2000–2023)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Target 2 — Figure 2c: Normalized to 100% ────────────────────────────────────
# For Viet Nam, normalize so that the three sectors sum to 100%
vn_total = vn[c_agri] + vn[c_ind] + vn[c_srv]
vn_agri_norm = vn[c_agri] / vn_total * 100
vn_ind_norm  = vn[c_ind]  / vn_total * 100
vn_srv_norm  = vn[c_srv]  / vn_total * 100

# Thailand already sums to 100%, but normalize for consistency
th_total = th[c_agri] + th[c_ind] + th[c_srv]
th_agri_norm = th[c_agri] / th_total * 100
th_ind_norm  = th[c_ind]  / th_total * 100
th_srv_norm  = th[c_srv]  / th_total * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, (yr, a, i, s, title) in zip(axes,
    [(vn['Year'], vn_agri_norm, vn_ind_norm, vn_srv_norm, 'Viet Nam'),
     (th['Year'], th_agri_norm, th_ind_norm, th_srv_norm, 'Thailand')]):
    ax.stackplot(yr, a, i, s,
                 labels=['Agriculture', 'Industry', 'Services'],
                 colors=[sector_colors['Agriculture'], sector_colors['Industry'],
                         sector_colors['Services']],
                 alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylim(0, 105)
    ax.set_xlim(years[0], years[-1])
    ax.grid(axis='y', alpha=0.3)

    # Annotate end percentages
    cumul = 0
    for val in [a.iloc[-1], i.iloc[-1], s.iloc[-1]]:
        ax.annotate(f'{val:.1f}%',
                    xy=(yr.iloc[-1], cumul + val / 2),
                    fontsize=8, ha='left', va='center', fontweight='bold')
        cumul += val

axes[0].set_ylabel('Share of GDP (%)')

# Shared legend at the bottom
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.suptitle('Figure 2c — Normalized Sectoral Composition of GDP (2000–2023)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Interpretation of Figures 2b and 2c:**

* **Figure 2b** makes the "hidden" component explicit. For Viet Nam from 2010 onward, the purple band at the top represents **Product taxes less subsidies on products** (~8–11% of GDP). This component is economically significant — it reflects net indirect taxation and grows with formalization of the economy. Thailand folds this into its sectoral figures, so no purple band appears.
* **Figure 2c** removes the compositional distortion by rescaling Viet Nam's three sectors to sum to 100%. This view is the most appropriate for **direct cross-country comparison** of sectoral structure, as it neutralizes the accounting methodology difference. Under this normalization, Viet Nam's 2023 structure is approximately 13% Agriculture / 42% Industry / 45% Services.

**Analysis & Conclusion:**

* **Viet Nam — Rapid Structural Transformation:** Agriculture's share declined sharply from ~25% in 2000 to ~12% by 2023 (raw) or ~13% (normalized), with the released share absorbed primarily by Industry and Services. This is the hallmark of an economy in active modernization — transitioning from agrarian production toward manufacturing and tertiary sectors.
* **Thailand — Mature Service Economy:** Thailand entered the period with agriculture already below 10% and a dominant Services sector (~55%). Its structure has remained relatively stable throughout 2000–2023, indicating that the major structural transition occurred before the observation window.
* **Methodological Divergence:** The ~8–11% gap in Viet Nam's raw data (Figure 2a) is not a data quality issue but a national accounting revision. Figure 2b reveals this "Product taxes less subsidies" component explicitly, while Figure 2c provides the cleanest basis for cross-country comparison by normalizing away the accounting difference.
* **Divergent Developmental Stages:** Under the normalized view (Figure 2c), Viet Nam in 2023 (~13% Agriculture / 42% Industry / 45% Services) is roughly at the structural stage Thailand occupied in the early 2000s (~9% / 37% / 55%), confirming a developmental lag of approximately one to two decades. Notably, Viet Nam's Industry share remains higher than Thailand's, reflecting its current role as a manufacturing-export hub.

### <span class="nb-only" data-num="3.1.3"></span>Target 3: Evaluate the standard of living

**Chart type:** Full-period line chart + zoomed recent comparison + gap trend

* **What:** GDP per capita, PPP (constant 2021 international $) for both countries over **2000–2023**, with a focused zoom on **2020–2023**.
* **Why:** The full-period panel preserves long-run context (level and convergence trajectory), while the zoomed panel provides a readable short-run comparison for the post-pandemic years. This avoids over-interpreting a short window while still highlighting recent movement.
* **How:** Top panel = full-period levels; middle panel = 2020–2023 levels with gap shading; bottom panel = 2020–2023 gap trend. Start/end labels are used for quick magnitude reading.

In [ ]:
# Target 3: Full-period context + zoomed recent comparison + recent gap trend
col = 'GDP per capita, PPP (constant 2021 international $)'

# Full period
full_years = vn['Year'].to_numpy()
vn_full = pd.to_numeric(vn[col], errors='coerce').to_numpy(dtype=float)
th_full = pd.to_numeric(th[col], errors='coerce').to_numpy(dtype=float)

# Recent window for focused comparison
vn_recent = vn[vn['Year'].between(2020, 2023)].reset_index(drop=True)
th_recent = th[th['Year'].between(2020, 2023)].reset_index(drop=True)
recent_years = vn_recent['Year'].to_numpy()
vn_recent_vals = pd.to_numeric(vn_recent[col], errors='coerce').to_numpy(dtype=float)
th_recent_vals = pd.to_numeric(th_recent[col], errors='coerce').to_numpy(dtype=float)
gap = th_recent_vals - vn_recent_vals

fig = plt.figure(figsize=(14, 9), constrained_layout=True)
gs = fig.add_gridspec(3, 1, height_ratios=[2.2, 1.6, 1.2])
ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])
ax2 = fig.add_subplot(gs[2], sharex=ax1)

# Panel 1: Full-period levels
shade_crises(ax0)
ln_vn0, = ax0.plot(full_years, vn_full, color=C_VN, marker='o', markersize=3.5, linewidth=2, label='Viet Nam')
ln_th0, = ax0.plot(full_years, th_full, color=C_TH, marker='s', markersize=3.5, linewidth=2, label='Thailand')
ax0.set_title('Living Standards (Full Period): GDP per Capita, PPP — Viet Nam vs Thailand (2000–2023)')
ax0.set_ylabel('GDP per capita, PPP (2021 int. $)')
ax0.set_xlim(full_years[0] - 0.5, full_years[-1] + 0.5)
ax0.grid(axis='y', alpha=0.3)

# Annotate start/end values on full panel
for vals, color, name in [(vn_full, C_VN, 'Viet Nam'), (th_full, C_TH, 'Thailand')]:
    for idx in [0, -1]:
        yr = full_years[idx]
        val = vals[idx]
        if name == 'Thailand' and idx == -1:
            offset = (-5, 5)
        elif name == 'Thailand':
            offset = (0, 8)
        else:
            offset = (0, -12)
        ax0.annotate(f'${val:,.0f}', xy=(yr, val), textcoords='offset points',
                     xytext=offset, ha='center', fontsize=8, color=color, fontweight='bold')

# Panel 2: Zoomed recent levels
shade_crises(ax1)
ax1.plot(recent_years, vn_recent_vals, color=C_VN, marker='o', markersize=5, linewidth=2.2)
ax1.plot(recent_years, th_recent_vals, color=C_TH, marker='s', markersize=5, linewidth=2.2)
ax1.fill_between(recent_years, vn_recent_vals, th_recent_vals, alpha=0.12, color='grey')
ax1.set_title('Zoomed View: GDP per Capita, PPP (2020–2023)')
ax1.set_ylabel('GDP per capita, PPP (2021 int. $)')
ax1.grid(axis='y', alpha=0.3)
ax1.set_xticks(recent_years)

# Panel 3: Recent gap trend
ln_gap, = ax2.plot(recent_years, gap, color='#555555', linewidth=2, marker='D', markersize=4, label='Gap (THL - VNM)')
ax2.fill_between(recent_years, gap, alpha=0.10, color='#555555')
ax2.set_xlabel('Year')
ax2.set_ylabel('Gap (int. $)')
ax2.grid(axis='y', alpha=0.3)
ax2.set_xlim(recent_years[0] - 0.2, recent_years[-1] + 0.2)
ax2.set_xticks(recent_years)

# Annotate start and end recent gap
for idx, ha in [(0, 'left'), (-1, 'right')]:
    ax2.annotate(f'${gap[idx]:,.0f}', xy=(recent_years[idx], gap[idx]),
                 fontsize=8, fontweight='bold', color='#555555', ha=ha,
                 textcoords='offset points', xytext=(12, -10))

fig.legend([ln_vn0, ln_th0, ln_gap], ['Viet Nam', 'Thailand', 'Gap (THL - VNM)'],
           loc='lower center', ncol=3, fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.02))

plt.show()

**Analysis & Conclusion:**

* **Long-run level difference is clear:** Across the full 2000–2023 period, Thailand remains above Viet Nam in GDP per capita (PPP).
* **Long-run convergence is also visible:** Viet Nam increases from about **37%** of Thailand's level in **2000** to about **64%** in **2023**, indicating substantial relative catch-up.
* **Recent window shows mild narrowing:** In the zoomed 2020–2023 panel, the absolute gap decreases from about **$8.1k** to about **$7.6k**.
* **Interpretation boundary:** The full-period panel supports long-run convergence, while the zoomed panel supports a recent narrowing over 2020–2023. The short-run pattern should be interpreted as a recent movement, not evidence of a continuous decline in the absolute gap for every year in the full period.

## <span class="nb-only" data-num="3.2"></span>Globalization

### <span class="nb-only" data-num="3.2.1"></span>Target 4: Assess the openness of the economy

**Chart type:** Dual-axis line chart (small multiples — one panel per country)

* **What:** Trade (% of GDP) and GDP growth (annual %) for each country, 2000–2023.
* **Why:** Trade openness ranges from ~100–200% while GDP growth ranges from ~-7–10%. These are on fundamentally different scales, making a dual axis necessary. Small multiples by country prevent the visual confusion of 4 overlapping lines. The dual-axis design is justified here because both metrics are percentages (same dimensionality), just at different magnitudes.
* **How:** Left y-axis: Trade (solid line, primary metric). Right y-axis: GDP growth (dashed line, secondary metric). Color encodes metric type within each panel. Pearson correlation annotated.

In [ ]:
# ── Target 4: Trade Openness vs GDP Growth ───────────────────────────────────────
c_trade = 'Trade (% of GDP)'
c_gdp = 'GDP growth (annual %)'

# Pre-compute shared axis limits across both countries
trade_min = min(vn[c_trade].min(), th[c_trade].min())
trade_max = max(vn[c_trade].max(), th[c_trade].max())
trade_pad = (trade_max - trade_min) * 0.08
gdp_min = min(vn[c_gdp].min(), th[c_gdp].min())
gdp_max = max(vn[c_gdp].max(), th[c_gdp].max())
gdp_pad = (gdp_max - gdp_min) * 0.08

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
trade_lines = []
gdp_lines = []

for ax, (data, title) in zip(axes, [(vn, 'Viet Nam'), (th, 'Thailand')]):
    shade_crises(ax)

    # Left axis: Trade
    ln1, = ax.plot(data['Year'], data[c_trade], color='#2C3E50', linewidth=2,
                   label='Trade (% of GDP)')
    ax.set_ylabel('Trade (% of GDP)', color='#2C3E50')
    ax.set_xlabel('Year')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlim(years[0], years[-1])
    ax.set_ylim(trade_min - trade_pad, trade_max + trade_pad)

    # Right axis: GDP growth
    ax2 = ax.twinx()
    ln2, = ax2.plot(data['Year'], data[c_gdp], color='#E74C3C', linewidth=2,
                    linestyle='--', label='GDP Growth (%)')
    ax2.axhline(0, color='#E74C3C', linewidth=0.5, alpha=0.4)
    ax2.set_ylabel('GDP Growth (%)', color='#E74C3C')
    ax2.set_ylim(gdp_min - gdp_pad, gdp_max + gdp_pad)

    if not trade_lines:
        trade_lines.append(ln1)
        gdp_lines.append(ln2)

    # Correlation annotation
    corr = data[[c_trade, c_gdp]].dropna().corr().iloc[0, 1]
    ax.text(0.98, 0.05, f'r = {corr:.2f}', transform=ax.transAxes,
            fontsize=10, ha='right', va='bottom',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.7))

    ax.grid(axis='y', alpha=0.3)

# Shared legend at the bottom — only the two metric lines
fig.legend(trade_lines + gdp_lines,
           ['Trade (% of GDP)', 'GDP Growth (%)'],
           loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.suptitle('Trade Openness vs GDP Growth (2000–2023)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Viet Nam — Rising Openness:** Viet Nam's trade-to-GDP ratio rises markedly over 2000-2023, reflecting deeper integration into global trade. Over the same period, GDP growth remains consistently positive, showing resilience even as openness increases.
* **Thailand — High Openness with Greater Volatility:** Thailand's trade openness stays high throughout the period but changes less dramatically than Viet Nam's. However, Thailand's GDP growth is much more volatile, with sharper downturns during crisis periods, especially in 2009 and 2020.
* **Correlation:** The historical trade-growth relationship is not especially strong. Viet Nam shows a weak negative correlation (`r ≈ -0.20`), while Thailand shows a modest positive correlation (`r ≈ 0.36`). This suggests the chart is more useful for comparing exposure and volatility than for claiming a direct causal link from trade openness to growth.


### <span class="nb-only" data-num="3.2.2"></span>Target 5: Measure global trade integration

**Chart type:** Line chart with trade-balance shading (small multiples)

* **What:** Exports and Imports (% of GDP) for each country, 2000–2023.
* **Why:** The analytical question is not just about magnitude, but about balance (exports minus imports). Two lines with conditional `fill_between` shading encode both the direction and magnitude of the trade balance in a single visual. Green fill = surplus, red fill = deficit. This is superior to a simple bar chart because it preserves temporal continuity.
* **How:** Position encodes magnitude (Cleveland rank 1). Color of fill encodes sign of trade balance (surplus vs deficit). Two panels prevent overplotting.

In [ ]:
# ── Target 5: Exports vs Imports ──────────────────────────────────────────────────
c_exp = 'Exports of goods and services (% of GDP)'
c_imp = 'Imports of goods and services (% of GDP)'

# Contrastive color palette: blue for exports, orange for imports
C_EXP      = '#1565C0'   # strong blue
C_IMP      = '#E65100'   # strong orange
C_SURPLUS  = '#90CAF9'   # light blue  (exports > imports)
C_DEFICIT  = '#FFAB91'   # light orange (imports > exports)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, (data, title) in zip(axes, [(vn, 'Viet Nam'), (th, 'Thailand')]):
    exp_vals = data[c_exp].values
    imp_vals = data[c_imp].values
    yrs = data['Year'].values

    ax.plot(yrs, exp_vals, color=C_EXP, linewidth=2, label='Exports')
    ax.plot(yrs, imp_vals, color=C_IMP, linewidth=2, linestyle='--', label='Imports')

    # Shade surplus (light blue) and deficit (light orange)
    ax.fill_between(yrs, exp_vals, imp_vals,
                    where=(exp_vals >= imp_vals), alpha=0.4, color=C_SURPLUS,
                    interpolate=True, label='Surplus')
    ax.fill_between(yrs, exp_vals, imp_vals,
                    where=(exp_vals < imp_vals), alpha=0.4, color=C_DEFICIT,
                    interpolate=True, label='Deficit')

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_xlim(years[0], years[-1])
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel('% of GDP')

# Shared legend at the bottom
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=10,
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.suptitle('Trade Composition: Exports vs Imports (2000–2023)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Viet Nam — From Deficit to Surplus:** Viet Nam transitioned from a trade deficit (imports exceeding exports) in the early 2000s to a roughly balanced or surplus position in recent years, reflecting the success of its export-oriented industrialization strategy.
* **Thailand — Consistent Surplus:** Thailand has maintained a consistent trade surplus throughout most of the period, with exports consistently exceeding imports. This reflects its position as a manufacturing and agricultural export hub.
* **Scale of Integration:** Both countries' export and import shares have increased over time, but Viet Nam's growth trajectory is notably steeper, consistent with the rising trade openness observed in Target 4.

### <span class="nb-only" data-num="3.2.3"></span>Target 6: Compare FDI and GDP fluctuation between countries

**Chart type:** Line chart (two rows — FDI and GDP growth — both countries overlaid)

* **What:** FDI net inflows (% of GDP) and GDP growth (%) for Viet Nam and Thailand on shared time axes, 2000–2023.
* **Why:** While Target 6a examines within-country FDI-GDP correlation, this view enables **cross-country comparison** of how the same two metrics fluctuate over time. Overlaying both countries on shared axes makes co-movement, divergence, and crisis response directly comparable (Cleveland rank 1: common-scale position comparison).
* **How:** Two-row layout: top = FDI, bottom = GDP growth. Each row plots both countries in their signature colors (red = VN, blue = TH). Crisis shading provides temporal context.

In [ ]:
# ── Target 6b: FDI & GDP Fluctuation Comparison ──────────────────────────────────
c_fdi = 'Foreign direct investment, net inflows (% of GDP)'
c_gdp = 'GDP growth (annual %)'

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                gridspec_kw={'hspace': 0.12})

# Top panel: FDI
shade_crises(ax1)
ax1.plot(years, vn[c_fdi], color=C_VN, marker='o', markersize=4, linewidth=2, label='Viet Nam')
ax1.plot(years, th[c_fdi], color=C_TH, marker='s', markersize=4, linewidth=2, label='Thailand')
ax1.set_ylabel('FDI, net inflows (% of GDP)')
ax1.set_title('FDI and GDP Growth Fluctuation: Viet Nam vs Thailand (2000–2023)', fontsize=14)
ax1.grid(axis='y', alpha=0.3)
ax1.axhline(0, color='black', linewidth=0.5, alpha=0.4)

# Annotate end values
for data, color in [(vn, C_VN), (th, C_TH)]:
    val = data[c_fdi].iloc[-1]
    ax1.annotate(f'{val:.1f}%', xy=(years[-1], val), fontsize=8, color=color,
                 fontweight='bold', textcoords='offset points', xytext=(5, 0))

# Bottom panel: GDP Growth
shade_crises(ax2)
ax2.plot(years, vn[c_gdp], color=C_VN, marker='o', markersize=4, linewidth=2, label='Viet Nam')
ax2.plot(years, th[c_gdp], color=C_TH, marker='s', markersize=4, linewidth=2, label='Thailand')
ax2.set_ylabel('GDP Growth (annual %)')
ax2.set_xlabel('Year')
ax2.grid(axis='y', alpha=0.3)
ax2.axhline(0, color='black', linewidth=0.5, alpha=0.4)
ax2.set_xlim(years[0] - 0.5, years[-1] + 0.5)

# Annotate end values
for data, color in [(vn, C_VN), (th, C_TH)]:
    val = data[c_gdp].iloc[-1]
    ax2.annotate(f'{val:.1f}%', xy=(years[-1], val), fontsize=8, color=color,
                 fontweight='bold', textcoords='offset points', xytext=(5, 0))

# Shared legend at the bottom
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.04), frameon=False)

plt.tight_layout()
plt.show()

**Analysis & Conclusion (6b):**

* **FDI pattern divergence:** Viet Nam records higher FDI inflows (% of GDP) than Thailand in many years, especially after ~2015. In this sample, the pattern is consistent with Viet Nam's stronger positioning in manufacturing-oriented investment flows.
* **GDP co-movement during crises:** Both countries' GDP growth declines during 2008–2009 and 2020, with deeper contractions observed for Thailand in those episodes.
* **Recovery profile differences:** After crisis years, Viet Nam's FDI series returns more quickly toward pre-shock levels in this window, while Thailand's recovery appears flatter. This should be interpreted as an observed pattern in the chart rather than proof of a single underlying cause.

### <span class="nb-only" data-num="3.2.4"></span>Target 7: Evaluate the transition from foreign aid to private investment

**Chart type:** 2×2 grid of area charts (rows: ODA, FDI; columns: Viet Nam, Thailand)

* **What:** Net ODA received (% of GNI) and FDI net inflows (% of GDP) for both countries, 2000–2023.
* **Why:** Separating ODA and FDI into dedicated rows avoids the scale-mixing problem of stacking two metrics with different denominators (% of GNI vs % of GDP). The 2×2 layout enables both within-country comparison (top vs bottom in same column) and cross-country comparison (left vs right in same row), following the small multiples principle (Tufte).
* **How:** Each cell shows a single metric for a single country as an area chart. Rows share y-axis limits within each metric. Country colors distinguish panels. Crisis shading provides temporal context.

In [ ]:
# ── Target 7: ODA vs FDI Transition (2×2 Grid) ───────────────────────────────────
c_oda = 'Net ODA received (% of GNI)'
c_fdi = 'Foreign direct investment, net inflows (% of GDP)'

import matplotlib.patheffects as pe

# Shared y-limits per metric across countries
oda_max = max(vn[c_oda].max(), th[c_oda].max()) * 1.15
fdi_max = max(vn[c_fdi].max(), th[c_fdi].max()) * 1.15

fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharex=True)

datasets = [(vn, 'Viet Nam', C_VN), (th, 'Thailand', C_TH)]

def annotate_edge_value(ax, x, y, text, color, position='end', dy=0):
    """Readable edge labels (no boxes), with white stroke for contrast."""
    if position == 'start':
        xytext, ha = (-8, dy), 'left'
    else:
        xytext, ha = (4, dy), 'right'

    ann = ax.annotate(
        text,
        xy=(x, y),
        textcoords='offset points',
        xytext=xytext,
        ha=ha,
        va='bottom',
        fontsize=10,
        fontweight='bold',
        color=color,
        zorder=6,
        clip_on=False,
    )
    ann.set_path_effects([
        pe.Stroke(linewidth=2.5, foreground='white'),
        pe.Normal(),
    ])

def add_anchor_dots(ax, x0, y0, x1, y1, color):
    """Emphasize annotated points so labels clearly map to real data points."""
    ax.scatter(
        [x0, x1], [y0, y1],
        s=44,
        color=color,
        edgecolors='white',
        linewidths=1.1,
        zorder=7,
    )

# Row 0: ODA
for j, (data, title, color) in enumerate(datasets):
    ax = axes[0, j]
    shade_crises(ax)
    ax.fill_between(data['Year'], data[c_oda].fillna(0), alpha=0.35, color=color)
    ax.plot(
        data['Year'], data[c_oda],
        color=color,
        linewidth=2,
        marker='o',
        markersize=3.5,
        markerfacecolor='white',
        markeredgecolor=color,
        markeredgewidth=0.9,
    )
    ax.set_title(f'{title} — ODA', fontsize=12, fontweight='bold')
    ax.set_ylim(0, oda_max)
    ax.set_xlim(years[0] - 0.8, years[-1] + 0.8)
    ax.grid(axis='y', alpha=0.3)

    # Annotate start & end with better contrast
    start_year = data['Year'].iloc[0]
    end_year = data['Year'].iloc[-1]
    start_val = data[c_oda].iloc[0]
    end_val = data[c_oda].iloc[-1]

    if not np.isnan(start_val) and not np.isnan(end_val):
        add_anchor_dots(ax, start_year, start_val, end_year, end_val, color)

    if not np.isnan(start_val):
        start_dy = 22 if start_val < 0.35 else 18
        annotate_edge_value(ax, start_year, start_val, f'{start_val:.2f}%', color, position='start', dy=start_dy)
    if not np.isnan(end_val):
        end_dy = 22 if end_val < 0.35 else 18
        annotate_edge_value(ax, end_year, end_val, f'{end_val:.2f}%', color, position='end', dy=end_dy)

axes[0, 0].set_ylabel('Net ODA (% of GNI)')

# Row 1: FDI
for j, (data, title, color) in enumerate(datasets):
    ax = axes[1, j]
    shade_crises(ax)
    ax.fill_between(data['Year'], data[c_fdi].fillna(0), alpha=0.35, color=color)
    ax.plot(
        data['Year'], data[c_fdi],
        color=color,
        linewidth=2,
        marker='o',
        markersize=3.5,
        markerfacecolor='white',
        markeredgecolor=color,
        markeredgewidth=0.9,
    )
    ax.set_title(f'{title} — FDI', fontsize=12, fontweight='bold')
    ax.set_ylim(0, fdi_max)
    ax.set_xlim(years[0] - 0.8, years[-1] + 0.8)
    ax.set_xlabel('Year')
    ax.grid(axis='y', alpha=0.3)

    # Annotate start & end with better contrast
    start_year = data['Year'].iloc[0]
    end_year = data['Year'].iloc[-1]
    start_val = data[c_fdi].iloc[0]
    end_val = data[c_fdi].iloc[-1]

    if not np.isnan(start_val) and not np.isnan(end_val):
        add_anchor_dots(ax, start_year, start_val, end_year, end_val, color)

    if not np.isnan(start_val):
        start_dy = 22 if start_val < 0.35 else 18
        annotate_edge_value(ax, start_year, start_val, f'{start_val:.1f}%', color, position='start', dy=start_dy)
    if not np.isnan(end_val):
        end_dy = 22 if end_val < 0.35 else 18
        annotate_edge_value(ax, end_year, end_val, f'{end_val:.1f}%', color, position='end', dy=end_dy)

axes[1, 0].set_ylabel('FDI, net inflows (% of GDP)')

# Shared legend at the bottom
legend_items = [Patch(facecolor=C_VN, alpha=0.5, label='Viet Nam'),
                Patch(facecolor=C_TH, alpha=0.5, label='Thailand')]
fig.legend(handles=legend_items, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.03), frameon=False)

plt.suptitle('External Financing Transition: ODA vs FDI (2000–2023)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Viet Nam — Clear Transition:** Viet Nam shows a textbook aid-to-investment transition. ODA as a share of GNI has steadily declined from a significant share in 2000 to near-zero by the early 2020s, while FDI has grown to dominate external financing. This reflects Viet Nam's graduation from low-income status and its increasing attractiveness to private investors.
* **Thailand — Already Post-Transition:** Thailand's ODA share was already negligible at the start of the period, consistent with its higher income level. FDI dominates entirely, with fluctuations reflecting global investment cycles rather than any aid-dependency dynamics.
* **Development Stage Marker:** The relative proportions of ODA vs FDI serve as a clear marker of economic maturity. Viet Nam's transition during this period mirrors the trajectory that Thailand completed in earlier decades.

## <span class="nb-only" data-num="3.3"></span>Environment

### <span class="nb-only" data-num="3.3.1"></span>Target 8: Track the historical trends of urban growth and forest cover

**Chart type:** Two-row line chart (top: Forest area, bottom: Urban pop. growth; both countries overlaid)

* **What:** Forest area (% of land) and Urban population growth (annual %) for both countries over time, 2000–2023.
* **Why:** While Target 8a examines the static correlation, this view tracks **how each metric evolved over time**, making trends, turning points, and cross-country divergences visible. Separating the two metrics into dedicated rows avoids the dual-axis perceptual distortion (stock vs flow on different scales), while overlaying both countries per row enables direct comparison (Cleveland rank 1).
* **How:** Top row: Forest area (stock). Bottom row: Urban growth (flow). Both countries plotted in signature colors. Crisis shading for context.

In [ ]:
# ── Target 8: Historical Tracking — Forest Area & Urban Growth ──────────────────
c_urban = 'Urban population growth (annual %)'
c_forest = 'Forest area (% of land area)'

import matplotlib.patheffects as pe

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(14, 7), sharex=True,
    gridspec_kw={'hspace': 0.12},
    constrained_layout=True
)

def annotate_end_value(ax, x, y, text, color, dy):
    """Readable end labels without boxed callouts."""
    ann = ax.annotate(
        text,
        xy=(x, y),
        xytext=(8, dy),
        textcoords='offset points',
        ha='left',
        va='center',
        fontsize=10,
        color=color,
        fontweight='bold',
        zorder=6,
        clip_on=False,
    )
    ann.set_path_effects([
        pe.Stroke(linewidth=2.5, foreground='white'),
        pe.Normal(),
    ])

# Top panel: Forest area (stock)
shade_crises(ax1)
ax1.plot(years, vn[c_forest], color=C_VN, marker='o', markersize=4, linewidth=2, label='Viet Nam')
ax1.plot(years, th[c_forest], color=C_TH, marker='s', markersize=4, linewidth=2, label='Thailand')
ax1.set_ylabel('Forest Area (% of land area)')
ax1.set_title('Forest Cover & Urban Growth Over Time (2000–2023)', fontsize=14)
ax1.grid(axis='y', alpha=0.3)

# End labels
annotate_end_value(ax1, years[-1], float(vn[c_forest].iloc[-1]), f"{vn[c_forest].iloc[-1]:.1f}%", C_VN, 6)
annotate_end_value(ax1, years[-1], float(th[c_forest].iloc[-1]), f"{th[c_forest].iloc[-1]:.1f}%", C_TH, 0)

# Bottom panel: Urban population growth (flow)
shade_crises(ax2)
ax2.plot(years, vn[c_urban], color=C_VN, marker='o', markersize=4, linewidth=2, label='Viet Nam')
ax2.plot(years, th[c_urban], color=C_TH, marker='s', markersize=4, linewidth=2, label='Thailand')
ax2.set_ylabel('Urban Pop. Growth (annual %)')
ax2.set_xlabel('Year')
ax2.grid(axis='y', alpha=0.3)
ax2.set_xlim(years[0] - 0.5, years[-1] + 1.0)

# End labels
annotate_end_value(ax2, years[-1], float(vn[c_urban].iloc[-1]), f"{vn[c_urban].iloc[-1]:.2f}%", C_VN, 6)
annotate_end_value(ax2, years[-1], float(th[c_urban].iloc[-1]), f"{th[c_urban].iloc[-1]:.2f}%", C_TH, -6)

# Shared legend at the bottom
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02), frameon=False)

plt.show()

**Analysis & Conclusion:**

* **Forest Area — Divergent Trajectories:** Viet Nam's forest cover shows an upward trend over the period, while Thailand's forest cover remains comparatively stable.
* **Urban Growth — General deceleration:** Both countries show a broad decline in urban population growth rates over time, with Thailand declining from a higher starting level.
* **Outlier caveat:** Thailand shows a pronounced one-year spike in urban growth (around 2018). This spike may reflect a definition/reclassification effect; interpretation is cautious.
* **Interpretation note:** The timing of Viet Nam's forest increase is consistent with policy-supported reforestation, but the chart alone should be interpreted as association rather than definitive causal evidence.

### <span class="nb-only" data-num="3.3.2"></span>Target 9: Assess drivers of deforestation

**Chart type:** Stacked area chart (2×1 small multiples — one panel per country)

* **What:** Forest area (% of land area) and Agricultural land (% of land area) for each country, 2000–2023.
* **Why:** Both metrics represent shares of total land area that can be meaningfully stacked to show combined land-use coverage. The stacked area chart reveals the part-to-whole relationship — how much of the land is allocated to forest vs agriculture — and whether expansion of one comes at the expense of the other. The remaining gap at the top represents other land uses (urban, water, etc.).
* **How:** Forest area stacked on the bottom (green), Agricultural land stacked on top (brown). Shared y-axis across panels for fair comparison. Annotations on end values.

In [ ]:
# ── Target 9: Forest & Agricultural Land Use (Stacked Area) ─────────────────────
c_forest = 'Forest area (% of land area)'
c_agland = 'Agricultural land (% of land area)'

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, (data, title) in zip(axes, [(vn, 'Viet Nam'), (th, 'Thailand')]):
    yrs = data['Year'].values
    forest_vals = data[c_forest].values
    agland_vals = data[c_agland].values

    ax.stackplot(yrs, forest_vals, agland_vals,
                 labels=['Forest area', 'Agricultural land'],
                 colors=['#2E7D32', '#8D6E63'], alpha=0.8)

    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_xlim(years[0], years[-1])
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)

    # Annotate end percentages
    f_end = forest_vals[-1]
    a_end = agland_vals[-1]
    ax.annotate(f'{f_end:.1f}%', xy=(yrs[-1], f_end / 2),
                fontsize=8, ha='left', va='center', fontweight='bold', color='white')
    ax.annotate(f'{a_end:.1f}%', xy=(yrs[-1], f_end + a_end / 2),
                fontsize=8, ha='left', va='center', fontweight='bold', color='white')

    # Annotate combined total at end
    total_end = f_end + a_end
    ax.annotate(f'Total: {total_end:.1f}%', xy=(yrs[-1], total_end),
                fontsize=8, ha='left', va='bottom', fontweight='bold',
                textcoords='offset points', xytext=(3, 5))

axes[0].set_ylabel('% of Land Area')

# Shared legend at the bottom
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.suptitle('Land Use Composition: Forest & Agricultural Land (2000–2023)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Viet Nam — Growing Forest, Stable Agriculture:** The stacked area reveals that Viet Nam's combined forest + agricultural land coverage has increased over time, driven primarily by forest area expansion (~37% → ~47%). Agricultural land has remained relatively stable, meaning forest recovery has come from reclaiming other land categories (barren land, degraded areas) rather than converting farmland — a sign of successful reforestation policy.
* **Thailand — Stable Composition:** Thailand's land-use composition has remained remarkably stable, with both forest (~37%) and agricultural land (~44%) showing minimal change. The combined coverage is higher (~80%), leaving less room for land-use shifts. This equilibrium suggests that Thailand's major land-use transitions occurred before 2000.
* **Decoupling Confirmed:** The stacked view makes it visually clear that agricultural expansion has **not** come at the expense of forest in either country during this period — challenging the classic deforestation-through-agriculture narrative for this specific time window.

## <span class="nb-only" data-num="3.4"></span>Healthcare & Demographics

### <span class="nb-only" data-num="3.4.1"></span>Target 10: Evaluate national prioritization of healthcare

**Chart type:** Grouped bar chart

* **What:** Current health expenditure (% of GDP) for Viet Nam and Thailand, 2000–2023.
* **Why:** A single metric compared across two entities over 23 time points. A grouped bar chart emphasizes discrete year-to-year values and makes magnitude comparison precise (Cleveland: length on a common scale). With only 2 groups per year, the chart remains legible without overcrowding.
* **How:** Side-by-side bars (position encodes magnitude). Color hue distinguishes countries. Consistent global color scheme (red = VN, blue = TH).

In [ ]:
# ── Target 10: Health Expenditure ──────────────────────────────────────────────────
c_health = 'Current health expenditure (% of GDP)'

fig, ax = plt.subplots(figsize=(14, 5))

w = 0.35
x = np.arange(len(years))

ax.bar(x - w/2, vn[c_health], width=w, color=C_VN, alpha=0.85, label='Viet Nam')
ax.bar(x + w/2, th[c_health], width=w, color=C_TH, alpha=0.85, label='Thailand')

ax.set_xlabel('Year')
ax.set_ylabel('Health Expenditure (% of GDP)')
ax.set_title('National Healthcare Prioritization: Health Expenditure as % of GDP (2000–2023)')
ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45, fontsize=8)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Thailand — Higher and Growing Commitment:** Thailand consistently spends a larger share of GDP on healthcare compared to Viet Nam. This reflects Thailand's Universal Coverage Scheme (introduced in 2002), which requires sustained public funding.
* **Viet Nam — Gradual Increase:** Viet Nam's health expenditure has been gradually increasing, suggesting growing national prioritization of healthcare. However, the gap with Thailand persists, indicating room for further investment.
* **Convergence Potential:** The trends suggest slow convergence, with Viet Nam's share rising more steeply. Whether this narrowing gap translates into improved health outcomes would require cross-referencing with mortality and life expectancy data.

### <span class="nb-only" data-num="3.4.2"></span>Target 11: Examine gender composition in national demographics

**Chart type:** Small multiples line chart (two rows, shared x-axis)

* **What:** Sex ratio at birth and Population female (%) for both countries, 2000–2023.
* **Why:** Two related but differently-scaled metrics. Both are demographic ratios with narrow ranges (sex ratio ~1.04–1.12, female % ~49–52%). Separate panels with aligned x-axes preserve perceptual accuracy without dual-axis distortion. Tight y-axis ranges reveal variation that would be invisible on a 0–100 scale.
* **How:** Top panel: Sex ratio at birth. Bottom panel: Female population %. Two lines per panel (VN, TH). Horizontal reference lines at biological/demographic baselines (1.05 for sex ratio, 50% for female share).

In [ ]:
# ── Target 11: Gender Demographics ────────────────────────────────────────────────
c_sex = 'Sex ratio at birth (male births per female births)'
c_fem = 'Population, female (% of total population)'

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel 1: Sex ratio at birth
ax = axes[0]
ax.plot(years, vn[c_sex], color=C_VN, marker='o', markersize=4, linewidth=2, label='Viet Nam')
ax.plot(years, th[c_sex], color=C_TH, marker='s', markersize=4, linewidth=2, label='Thailand')
ax.axhline(1.05, color='black', linewidth=1, linestyle=':', alpha=0.6, label='Biological norm (1.05)')
ax.set_ylabel('Male births per female birth')
ax.set_title('Sex Ratio at Birth')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Panel 2: Female population share
ax = axes[1]
ax.plot(years, vn[c_fem], color=C_VN, marker='o', markersize=4, linewidth=2, label='Viet Nam')
ax.plot(years, th[c_fem], color=C_TH, marker='s', markersize=4, linewidth=2, label='Thailand')
ax.axhline(50, color='black', linewidth=1, linestyle=':', alpha=0.6, label='Parity (50%)')
ax.set_ylabel('Female (% of total pop.)')
ax.set_xlabel('Year')
ax.set_title('Female Share of Total Population')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_xlim(years[0] - 0.5, years[-1] + 0.5)

plt.suptitle('Gender Composition in National Demographics (2000–2023)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Viet Nam — Elevated Sex Ratio:** Viet Nam's sex ratio at birth is consistently above the biological norm of 1.05, and has risen over the period, suggesting the influence of cultural son preference. This is a well-documented phenomenon in Vietnamese demographics.
* **Thailand — Near Biological Norm:** Thailand's sex ratio at birth remains close to the natural 1.05–1.06 range throughout, indicating minimal sex-selective practices.
* **Female Population Share:** Both countries maintain female population shares above 50%, which is consistent with the biological pattern of higher male infant mortality and shorter male life expectancy. Viet Nam's slightly higher female share, combined with its elevated sex ratio at birth, suggests differential mortality is compensating for birth-level imbalances.

### <span class="nb-only" data-num="3.4.3"></span>Target 12: Analyze the national demographic transition

**Chart type:** Line chart with natural-increase shading (small multiples)

* **What:** Birth rate and Death rate (crude, per 1,000 people) for each country, 2000–2023.
* **Why:** Both metrics share the same unit (per 1,000), so a common y-axis is valid and preferred over dual axes. The gap between birth and death rates represents the rate of natural population increase — a key demographic indicator. Using `fill_between` to shade this gap encodes an analytically meaningful derived quantity without adding chartjunk.
* **How:** Solid line = Birth rate. Dashed line = Death rate. Shaded area between = natural increase. Two panels by country. Consistent y-axis for cross-country comparison.

In [ ]:
# ── Target 12: Demographic Transition ─────────────────────────────────────────────
c_birth = 'Birth rate, crude (per 1,000 people)'
c_death = 'Death rate, crude (per 1,000 people)'

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, (data, title, color) in zip(axes, [(vn, 'Viet Nam', C_VN), (th, 'Thailand', C_TH)]):
    yrs = data['Year'].values
    birth = data[c_birth].values
    death = data[c_death].values
    
    ax.plot(yrs, birth, color=color, linewidth=2, label='Birth Rate')
    ax.plot(yrs, death, color=color, linewidth=2, linestyle='--', alpha=0.7, label='Death Rate')
    ax.fill_between(yrs, birth, death, alpha=0.15, color=color, label='Natural Increase')
    
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_xlim(years[0], years[-1])
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8)
    
    # Annotate gap at start and end
    for idx in [0, -1]:
        gap = birth[idx] - death[idx]
        mid = (birth[idx] + death[idx]) / 2
        ax.annotate(f'gap={gap:.1f}', xy=(yrs[idx], mid), fontsize=8,
                    fontweight='bold', ha='center',
                    textcoords='offset points', xytext=(0, 0))

axes[0].set_ylabel('Rate per 1,000 people')
plt.suptitle('Demographic Transition: Birth Rate vs Death Rate (2000–2023)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Both Countries — Declining Birth Rates:** Both Viet Nam and Thailand show declining birth rates over the period, consistent with the demographic transition model. Thailand's decline started from a lower base, reflecting its more advanced demographic stage.
* **Converging Gap:** The shaded area (natural increase) is narrowing for both countries, indicating slowing population growth. Thailand's gap is narrower, suggesting it is further along the transition toward population stabilization.
* **Thailand — Approaching Equilibrium:** Thailand's birth and death rates are converging more rapidly, raising the possibility of future population decline if trends continue. This has significant implications for labor force planning and dependency ratios.
* **Viet Nam — Still Growing:** Viet Nam maintains a larger gap between birth and death rates, indicating continued natural population growth, though at a decelerating pace.

## <span class="nb-only" data-num="3.5"></span>Labor & Workforce

### <span class="nb-only" data-num="3.5.1"></span>Target 13: Identify gender disparity in economic participation

**Chart type:** Line chart with gender-gap shading (small multiples)

* **What:** Labor force participation rates for female and male populations in each country, 2000–2023.
* **Why:** The analytical focus is the *gap* between male and female participation — not the absolute levels. A line chart with `fill_between` for the gender gap directly encodes the disparity magnitude as a shaded ribbon. A wider ribbon = larger disparity. This immediately shows whether the gap is narrowing over time.
* **How:** Male line on top, female line below. Shaded ribbon between them. One panel per country. Gap values annotated at start and end years for precise reading.

In [ ]:
# ── Target 13: Gender Labor Gap ───────────────────────────────────────────────────
c_fem_labor = 'Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)'
c_mal_labor = 'Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)'

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, (data, title, color) in zip(axes, [(vn, 'Viet Nam', C_VN), (th, 'Thailand', C_TH)]):
    yrs = data['Year'].values
    male = data[c_mal_labor].values
    female = data[c_fem_labor].values
    
    ax.plot(yrs, male, color=color, linewidth=2, label='Male')
    ax.plot(yrs, female, color=color, linewidth=2, linestyle='--', alpha=0.7, label='Female')
    ax.fill_between(yrs, male, female, alpha=0.15, color=color, label='Gender Gap')
    
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_xlim(years[0], years[-1])
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8)
    
    # Annotate gap at start and end
    for idx, ha in [(0, 'left'), (-1, 'right')]:
        gap = male[idx] - female[idx]
        mid = (male[idx] + female[idx]) / 2
        ax.annotate(f'Gap: {gap:.1f}pp', xy=(yrs[idx], mid), fontsize=8,
                    fontweight='bold', ha=ha, color=color,
                    textcoords='offset points', xytext=(5 if ha == 'left' else -5, 0))

axes[0].set_ylabel('Participation Rate (%)')
plt.suptitle('Gender Disparity in Labor Force Participation (2000–2023)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Viet Nam — Narrow and Stable Gap:** Viet Nam shows a relatively small gender gap in labor force participation, with both male and female rates at high levels. This reflects Viet Nam's historical pattern of high female economic participation, influenced by cultural and economic necessity.
* **Thailand — Wider Gap:** Thailand exhibits a larger gap between male and female participation rates. While both have declined slightly over time, the gender disparity has remained relatively persistent.
* **Declining Overall Participation:** Both countries show a gradual decline in overall participation rates (both male and female), likely reflecting rising education enrollment, later workforce entry, and population aging effects.
* **Structural Interpretation:** Viet Nam's narrower gap does not necessarily indicate better gender equality in *quality* of employment — it may partly reflect economic necessity driving female labor force entry in lower-wage sectors.

### <span class="nb-only" data-num="3.5.2"></span>Target 14: Analyze the structural shift in employment across economic sectors

**Chart type:** Stacked area chart (1×2 small multiples — one panel per country)

* **What:** Employment shares in Agriculture, Industry, and Services (% of total employment) for each country, 2000–2023.
* **Why:** Employment composition reveals how labor migrates between sectors during economic development. While GDP composition (Target 2) shows *value* shifts, employment shows *people* shifts — the two can diverge significantly if productivity differs across sectors.
* **How:** Stacked area charts show part-to-whole composition over time. Using the same chart type as Target 2 (GDP composition) enables direct visual comparison of structural transformation in output vs. labor allocation.

In [ ]:
# ── Target 14: Employment Structure by Sector (Stacked Area) ─────────────────────
c_emp_agri = 'Employment in agriculture (% of total employment) (modeled ILO estimate)'
c_emp_ind  = 'Employment in industry (% of total employment) (modeled ILO estimate)'
c_emp_srv  = 'Employment in services (% of total employment) (modeled ILO estimate)'

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

colors_emp = ['#2E7D32', '#1565C0', '#F57C00']   # agriculture green, industry blue, services orange
labels_emp = ['Agriculture', 'Industry', 'Services']

for ax, (data, title) in zip(axes, [(vn, 'Viet Nam'), (th, 'Thailand')]):
    yrs = data['Year'].values
    agri = data[c_emp_agri].values
    ind  = data[c_emp_ind].values
    srv  = data[c_emp_srv].values

    ax.stackplot(yrs, agri, ind, srv, colors=colors_emp, alpha=0.85, labels=labels_emp)
    shade_crises(ax)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_xlim(years[0], years[-1])
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)

axes[0].set_ylabel('Share of Total Employment (%)')

handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.04), frameon=False)

plt.suptitle('Employment Composition by Sector (2000–2023)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Viet Nam — Rapid Agricultural Exit:** In 2000, over 60% of Viet Nam's workforce was employed in agriculture. By 2023, this has dropped to roughly 27%, representing one of the fastest labor-structural transformations in Southeast Asia. The released labor has been absorbed almost equally by industry and services, reflecting Viet Nam's dual strategy of export-oriented manufacturing and domestic service-sector growth.

* **Thailand — Services-Led Transition:** Thailand's agricultural employment share has also declined (from ~44% to ~31%), but the shift has been predominantly toward services rather than industry. Thailand's industrial employment share has remained relatively flat, suggesting that Thailand's manufacturing sector has grown through productivity gains rather than labor absorption.

* **Contrast with GDP Composition (Target 2):** Comparing employment shares with GDP shares reveals a productivity gap: agriculture still employs a disproportionately large share of workers relative to its GDP contribution in both countries, indicating lower labor productivity in agriculture compared to industry and services. This structural mismatch is more pronounced in Viet Nam.

* **Crisis Resilience:** Both countries show minimal disruption to employment composition during the 2008 and 2020 crises, suggesting that sectoral labor shifts are driven by long-term structural forces rather than cyclical shocks.

### <span class="nb-only" data-num="3.5.3"></span>Target 15: Assess the macroeconomic drivers of unemployment

**Chart type:** Multi-line chart (small multiples)

* **What:** Unemployment (%), Inflation (%), and GDP growth (%) for each country, 2000–2023.
* **Why:** All three metrics are annual percentages on broadly comparable scales, making a single y-axis valid and avoiding the perceptual distortions of dual axes. Three lines per panel with distinct visual styles enable simultaneous tracking of the Phillips Curve (unemployment–inflation trade-off) and Okun's Law (growth–unemployment link).
* **How:** Distinct line styles: solid = GDP growth, dashed = Inflation, dotted = Unemployment. Consistent color per metric across both panels (green, orange, red). A "stability zone" (0–5%) shaded for reference.

In [ ]:
# ── Target 15: Macroeconomic Drivers of Unemployment ──────────────────────────────
c_unemp = 'Unemployment, total (% of total labor force) (modeled ILO estimate)'
c_infl = 'Inflation, consumer prices (annual %)'
c_gdp = 'GDP growth (annual %)'

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, (data, title) in zip(axes, [(vn, 'Viet Nam'), (th, 'Thailand')]):
    yrs = data['Year'].values

    # Stability zone
    ax.axhspan(0, 5, color='#E8F5E9', alpha=0.5, zorder=0)
    shade_crises(ax)

    ax.plot(yrs, data[c_gdp], color='#27AE60', linewidth=2, label='GDP Growth')
    ax.plot(yrs, data[c_infl], color='#F39C12', linewidth=2, linestyle='--', label='Inflation')
    ax.plot(yrs, data[c_unemp], color='#E74C3C', linewidth=2, linestyle=':', label='Unemployment')

    ax.axhline(0, color='black', linewidth=0.5, alpha=0.4)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_xlim(years[0], years[-1])
    ax.grid(axis='y', alpha=0.3)
    ax.text(0.02, 0.02, 'Stability Zone (0–5%)', transform=ax.transAxes,
            fontsize=7, color='green', alpha=0.6, va='bottom')

axes[0].set_ylabel('Annual Rate (%)')

# Shared legend at the bottom
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.suptitle('Macroeconomic Stability: Growth, Inflation & Unemployment (2000–2023)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Analysis & Conclusion:**

* **Viet Nam — Inflation Volatility:** Viet Nam experienced significant inflation spikes (notably around 2008 and 2011), while GDP growth remained relatively stable and unemployment stayed low. The decoupling of unemployment from inflation suggests a labor market dominated by informal employment that does not respond to Phillips Curve dynamics in the classical sense.
* **Thailand — Low Unemployment Puzzle:** Thailand maintains remarkably low unemployment throughout, even during crisis periods when GDP growth turned sharply negative. This is characteristic of economies with large informal sectors where workers shift to subsistence activities rather than becoming formally unemployed.
* **Okun's Law Test:** The expected inverse relationship between GDP growth and unemployment is weak in both countries, likely because formal unemployment statistics undercount true labor market slack in developing economies.
* **Post-2015 Convergence:** Both countries show all three metrics converging toward the 0–5% stability zone in recent years, suggesting increasing macroeconomic maturity.

# <span class="nb-only" data-num="4."></span>Conclusion

This section synthesizes the findings from all 15 analysis targets, organized by domain, to provide a holistic comparative assessment of Viet Nam and Thailand's development trajectories from 2000 to 2023.

## <span class="nb-only" data-num="4.1"></span>Macroeconomics (Targets 1–3)

Viet Nam and Thailand represent two distinct stages of economic development that are converging over time. Viet Nam's GDP growth has been remarkably consistent and crisis-resistant, maintaining positive rates even during the 2008 and 2020 shocks — a resilience that Thailand, with its deeper global financial integration, has not matched. The structural composition analysis confirms that Viet Nam is undergoing a rapid transformation from agriculture to industry and services, a transition Thailand had largely completed before 2000. In PPP-adjusted per-capita terms, Viet Nam has roughly tripled its living standard over the period while steadily closing the gap with Thailand, though a significant disparity remains. Together, these three targets paint a picture of Viet Nam as a high-growth, rapidly modernizing economy that is following — and compressing — the developmental path Thailand traversed in earlier decades.

## <span class="nb-only" data-num="4.2"></span>Globalization (Targets 4–7)

Both countries are deeply integrated into the global economy, but the nature and trajectory of that integration differ markedly. Viet Nam's trade-to-GDP ratio has surged dramatically, driven by a successful export-oriented industrialization strategy that has shifted the country from trade deficit to surplus. FDI plays a structurally central role in Viet Nam's growth model, with a tighter FDI-growth correlation compared to Thailand's more diversified economic base. Perhaps most strikingly, the ODA-to-FDI transition reveals Viet Nam's graduation from aid dependence to private investment magnet — a textbook development-finance evolution that Thailand completed in an earlier era. Thailand's trade profile is more mature and stable, but its higher crisis sensitivity suggests that established integration carries its own vulnerabilities.

## <span class="nb-only" data-num="4.3"></span>Environment (Targets 8–9)

The environmental analysis shows differentiated but generally stable land-use dynamics over 2000–2023. In this window, Viet Nam's forest area increases while Thailand's forest share remains broadly flat. At the same time, urban population growth decelerates in both countries, suggesting slower marginal urban expansion in later years. The agriculture–forest comparison indicates no clear evidence in this period that agricultural expansion systematically displaced forest area in either country. Overall, the observed pattern is one of relative land-use stabilization, with stronger forest recovery visible in Viet Nam than in Thailand.

## <span class="nb-only" data-num="4.4"></span>Healthcare & Demographics (Targets 10–12)

Thailand allocates a consistently larger share of GDP to healthcare than Viet Nam, reflecting its Universal Coverage Scheme introduced in 2002. Viet Nam's health expenditure is growing but remains lower, suggesting room for expansion as the economy matures. Demographically, the two countries diverge in gender composition: Viet Nam's sex ratio at birth is notably elevated above the biological norm, pointing to cultural son-preference practices, while Thailand's remains near natural levels. Both countries are advancing through the demographic transition, with declining birth rates and narrowing natural increase gaps. Thailand is further along this trajectory and approaching population equilibrium, raising future concerns about workforce shrinkage, while Viet Nam maintains a larger demographic growth buffer.

## <span class="nb-only" data-num="4.5"></span>Labor & Workforce (Targets 13–15)

Viet Nam exhibits a notably narrow gender gap in labor force participation — a structural feature that distinguishes it from Thailand, where the gap is larger and more persistent. However, this narrow gap may partly reflect economic necessity rather than progressive gender policy. The employment structure analysis (Target 14) reinforces the structural-shift narrative from Target 2: Viet Nam's labor force has undergone a dramatic exodus from agriculture (64% → ~27%), with workers absorbed into both industry and services. Thailand's transition is more services-led, with industrial employment remaining flat — suggesting productivity-driven growth rather than labor absorption. Notably, agriculture still employs a disproportionately large share of workers relative to its GDP contribution in both countries, highlighting a persistent sectoral productivity gap.

The macroeconomic drivers analysis reveals that both countries defy classical Phillips Curve expectations: unemployment remains remarkably low and stable regardless of inflation or growth volatility. This is characteristic of economies with large informal sectors, where workers shift to subsistence activities rather than appearing in unemployment statistics. Post-2015, all three macroeconomic indicators (growth, inflation, unemployment) have converged toward a 0–5% stability zone for both countries, suggesting increasing macroeconomic maturity and institutional sophistication.

---

**Overall:** Viet Nam and Thailand represent a compelling case study in developmental convergence. Viet Nam is compressing into two decades the structural transformations that took Thailand considerably longer. While Thailand leads in absolute income levels, institutional maturity, and healthcare investment, Viet Nam's growth trajectory, crisis resilience, and rapid structural change suggest it is on course to narrow these gaps significantly in the coming decades. The key risks highlighted in this notebook include Viet Nam's elevated sex ratio at birth, Thailand's aging-demographic pressure, and each country's distinct exposure to external shocks and labor-structure adjustment challenges.